# Equalyze API Demo: CI/CD Integration

This notebook demonstrates how easily the **Equalyze Python SDK** can be integrated into an ML workflow. It showcases an **"Active AI Governance"** approach, where algorithms are audited for fairness *before* deployment, and blocked if they exhibit severe bias.

We will:
1. Setup the Equalyze environment.
2. Generate a biased loan approval dataset.
3. Run the Equalyze SDK as a simulated CI/CD gate to detect the bias and block the deployment.

## Step 1: Setup & Environment

First, we install the necessary dependencies including the `equalyze` SDK.

In [ ]:
!pip install equalyze pandas numpy

### Authentication & Configuration

Enter your Equalyze API Key and the API Base URL below. If you're using Colab Secrets, you can retrieve it automatically.

In [ ]:
import os
from google.colab import userdata

# Attempt to load from Colab Secrets, otherwise fallback to the dev bypass token.
try:
    os.environ["EQUALYZE_API_KEY"] = userdata.get('EQUALYZE_API_KEY')
except:
    os.environ["EQUALYZE_API_KEY"] = "DEV_MOCK_TOKEN"

# Set the API backend URL. Note: If your frontend proxies the API, you can use the frontend URL.
os.environ["EQUALYZE_BASE_URL"] = "https://equalyze-frontend-1085178935109.us-central1.run.app"

print(f"Configured to connect to: {os.environ['EQUALYZE_BASE_URL']}")

## Step 2: Generating a Biased Dataset

We generate 1,000 synthetic records mimicking a loan application system. The generation script inherently discriminates against **Female** applicants, specifically those from **Rural** zip codes. This ensures that the Equalyze fairness metrics will detect Disparate Impact and block the process.

In [ ]:
import pandas as pd
import numpy as np

def generate_biased_data(num_samples: int = 1000, output_path: str = "predictions.csv"):
    np.random.seed(42)
    genders = np.random.choice(['Male', 'Female'], size=num_samples, p=[0.6, 0.4])
    regions = np.random.choice(['Urban', 'Rural'], size=num_samples, p=[0.7, 0.3])
    incomes = np.clip(np.random.normal(loc=60000, scale=20000, size=num_samples), 20000, 200000)
    credit_scores = np.clip(np.random.normal(loc=650, scale=50, size=num_samples), 300, 850)
    
    df = pd.DataFrame({
        'applicant_id': range(1, num_samples + 1),
        'gender': genders,
        'region': regions,
        'income': incomes.astype(int),
        'credit_score': credit_scores.astype(int)
    })
    
    base_risk = (850 - df['credit_score']) / 550 + (200000 - df['income']) / 360000
    df['true_default_risk'] = np.clip(base_risk, 0.1, 0.9)
    
    approval_probabilities = 1.0 - df['true_default_risk']
    # Penalize Female and Rural Female
    approval_probabilities = np.where(df['gender'] == 'Female', approval_probabilities * 0.5, approval_probabilities)
    approval_probabilities = np.where((df['gender'] == 'Female') & (df['region'] == 'Rural'), approval_probabilities * 0.4, approval_probabilities)
    
    df['loan_approved'] = np.random.binomial(1, np.clip(approval_probabilities, 0.0, 1.0))
    df['loan_approved'] = df['loan_approved'].map({1: 'Approved', 0: 'Rejected'})
    df = df.drop(columns=['true_default_risk'])
    df.to_csv(output_path, index=False)
    print(f"Generated biased dataset at {output_path}.")
    return df

df = generate_biased_data()
df.head()

## Step 3: CI/CD Simulation using Equalyze SDK

This represents a deployment pipeline. The pipeline attempts to deploy the ML model, but Equalyze intercepts the DataFrame directly, audits it, and fails the CI/CD job gracefully if it breaches compliance thresholds.

In [ ]:
import sys
from equalyze import EqualyzeClient
from equalyze.exceptions import EqualyzeAPIError, EqualyzeTimeoutError

def run_fairness_gate(dataframe):
    print("--- [CI/CD Pipeline] Equalyze Fairness Gate ---")
    
    try:
        # The client automatically picks up EQUALYZE_API_KEY from the environment
        client = EqualyzeClient()
        
        print("\nUploading Pandas DataFrame...")
        # Using the convenient upload_dataframe method!
        upload_resp = client.datasets.upload_dataframe(dataframe)
        
        print("\nInitiating Fairness Audit...")
        result = client.audits.run(
            dataset_id=upload_resp.dataset_id, 
            protected_attributes=["gender", "region"], 
            outcome="loan_approved",
            threshold=0.85,
            timeout_sec=120
        )
        
        print("\n--- CI/CD STATUS REPORT ---")
        if result.overall_score < 0.85 or result.overall_severity in ["AMBER", "RED"]:
            print("Status: [BLOCKED] Exit Code 1")
            print(f"Details: Fairness score {result.overall_score} fell below the threshold. Pipeline halted to prevent deployment.")
        else:
            print("Status: [PASSED] Exit Code 0")
            print(f"Details: Fairness score {result.overall_score} meets threshold limits.")
            
    except EqualyzeTimeoutError as e:
        print("\n--- CI/CD STATUS REPORT ---")
        print("Status: [ERROR] Exit Code 2")
        print(f"Details: Audit timed out: {e}")
    except EqualyzeAPIError as e:
        print("\n--- CI/CD STATUS REPORT ---")
        print("Status: [ERROR] Exit Code 2")
        print(f"Details: API/Auth Error: {e}")

# Run the simulation
run_fairness_gate(df)